# M04B Scientific Screening Completion, Exploratory Statistics & Finalist Selection (`M04B_20261127_164911_DESKTOP-CQB09OU`)

**Scope:** Analysis-only notebook consuming saved M04A/M04B artifacts (`Protocol A` `validation=3,060` rows / `510` gridline groups, `seed=42`).
**Governance:** Does NOT retrain models, run HPO, access historical `test` (`3,822` rows), or run robustness sweeps. Confidence intervals are **exploratory** (`D-010C`).

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
RUN_ID = "M04B_20261127_164911_DESKTOP-CQB09OU"
RUN_DIR = ROOT / "reports" / "runs" / RUN_ID
FIG_DIR = RUN_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 140, "axes.grid": True, "grid.alpha": 0.25})
print("M04B Run Dir:", RUN_DIR)

## 1–5. Governance, M04A Starting State & Recovery Outcomes

In [ ]:
m04a_verif = json.loads((RUN_DIR / "M04A_EVIDENCE_VERIFICATION.json").read_text(encoding="utf-8"))
rec_attempts = json.loads((RUN_DIR / "M04B_RECOVERY_ATTEMPTS.json").read_text(encoding="utf-8"))
status_df = pd.read_csv(RUN_DIR / "M04B_MODEL_STATUS.csv")
print("M04A verified PASS:", m04a_verif["verified_pass_count"], "/", m04a_verif["expected_pass_count"])
print("Recovery attempts:")
for mid, info in rec_attempts.items():
    print(f"  {mid}: {info['status']} (wall={info['runner_summary'].get('elapsed_sec')}s, err={info['runner_summary'].get('violation_reason')})")
print("\n32-Model Status Summary:\n", status_df["scientific_status"].value_counts())

## 6–8. Consolidated Raw Leaderboard, Nonnegative-Clipped Leaderboard & Historical DLM/AED

In [ ]:
raw_m = pd.read_csv(RUN_DIR / "M04B_CONSOLIDATED_METRICS_RAW.csv")
clip_m = pd.read_csv(RUN_DIR / "M04B_CONSOLIDATED_METRICS_NONNEGATIVE.csv")
ref_m = pd.read_csv(RUN_DIR / "M04B_REFERENCE_VALIDATION_METRICS.csv")
pred_all = pd.read_csv(RUN_DIR / "M04B_CONSOLIDATED_VALIDATION_PREDICTIONS.csv")

cols_show = ["rank_raw_mae", "model_id", "mae_um", "rmse_um", "lins_ccc", "r2", "bias_um", "within_20um_rate"]
print("=== CONSOLIDATED RAW LEADERBOARD (D-010E PRIMARY) ===")
print(raw_m[cols_show].to_string(index=False))
print("\n=== NONNEGATIVE-CLIPPED LEADERBOARD (SEPARATE) ===")
print(clip_m[["rank_clipped_mae", "model_id", "mae_um", "rmse_um", "lins_ccc", "r2"]].to_string(index=False))
print("\n=== HISTORICAL REFERENCES (SEPARATE PROVENANCE) ===")
print(ref_m[["model_id", "mae_um", "rmse_um", "lins_ccc", "r2", "provenance"]].to_string(index=False))

## 9–13. Actual vs Predicted, Error Distributions, Bland-Altman, Tolerances & Negative Predictions

In [ ]:
top4 = raw_m["model_id"].head(4).tolist()
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, mid in zip(axes.flat, top4):
    sub = pred_all[pred_all["model_id"] == mid]
    ax.scatter(sub["y_true_um"], sub["y_pred_raw_um"], s=6, alpha=0.35)
    lim = max(sub["y_true_um"].max(), sub["y_pred_raw_um"].max())
    ax.plot([0, lim], [0, lim], "r--", lw=1)
    mae_v = raw_m.loc[raw_m["model_id"] == mid, "mae_um"].values[0]
    ax.set_title(f"{mid} (MAE={mae_v:.2f} um, n={len(sub)})")
    ax.set_xlabel("Actual MANwidth (um)")
    ax.set_ylabel("Predicted Raw (um)")
plt.suptitle(f"M04B Actual vs Predicted — Top-4 ({RUN_ID})")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig01_actual_vs_predicted_top4.png")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, mid in zip(axes.flat, top4):
    sub = pred_all[pred_all["model_id"] == mid]
    ax.hist(sub["raw_signed_error_um"], bins=50, color="#2ca02c", alpha=0.75)
    ax.axvline(0.0, color="k", ls="--", lw=1)
    ax.set_title(f"{mid} Signed Error (Pred - True)")
    ax.set_xlabel("Signed Error (um)")
plt.suptitle(f"M04B Error Distributions — Top-4 ({RUN_ID})")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig02_error_distributions_top4.png")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, mid in zip(axes.flat, top4):
    sub = pred_all[pred_all["model_id"] == mid]
    mv = 0.5 * (sub["y_true_um"] + sub["y_pred_raw_um"])
    dv = sub["y_pred_raw_um"] - sub["y_true_um"]
    md_v, sd_v = dv.mean(), dv.std(ddof=1)
    ax.scatter(mv, dv, s=6, alpha=0.3)
    ax.axhline(md_v, color="r", lw=1.2, label=f"Mean={md_v:.2f}")
    ax.axhline(md_v + 1.96 * sd_v, color="b", ls="--", lw=1, label=f"+1.96 SD={md_v+1.96*sd_v:.1f}")
    ax.axhline(md_v - 1.96 * sd_v, color="b", ls="--", lw=1, label=f"-1.96 SD={md_v-1.96*sd_v:.1f}")
    ax.set_title(f"{mid} Descriptive Bland-Altman (Row-Level)")
    ax.set_xlabel("Mean (um)")
    ax.set_ylabel("Diff (um)")
    ax.legend(fontsize=7)
plt.suptitle(f"M04B Descriptive Bland-Altman ({RUN_ID})")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig03_bland_altman_top4.png")
plt.show()

fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(raw_m))
w = 0.2
for idx, col in enumerate(["within_5um_rate", "within_10um_rate", "within_20um_rate", "within_50um_rate"]):
    ax.bar(x + (idx - 1.5) * w, raw_m[col] * 100.0, width=w, label=col.replace("_rate", ""))
ax.set_xticks(x)
ax.set_xticklabels(raw_m["model_id"], rotation=45, ha="right")
ax.set_ylabel("Hit Rate (%)")
ax.set_title(f"M04B Analytical Tolerance Hit Rates ({RUN_ID}, n=3,060)")
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "fig04_tolerance_hit_rates.png")
plt.show()

neg_df = pd.read_csv(RUN_DIR / "M04B_NEGATIVE_PREDICTION_ANALYSIS.csv")
print("=== NEGATIVE RAW PREDICTION ANALYSIS ===")
print(neg_df[neg_df["n_negative_raw_predictions"] > 0].to_string(index=False))

## 14–15. Exploratory Image-Cluster Bootstrap CIs & Paired MAE Difference Forest Plot

In [ ]:
boot_m = pd.read_csv(RUN_DIR / "M04B_BOOTSTRAP_MODEL_INTERVALS.csv")
boot_p = pd.read_csv(RUN_DIR / "M04B_BOOTSTRAP_PAIRED_MAE_DIFFERENCES.csv")
order_ids = raw_m["model_id"].tolist()
bm_plot = boot_m.set_index("model_id").loc[order_ids].reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(bm_plot))
ax.errorbar(
    bm_plot["mae_point_um"], y_pos,
    xerr=[bm_plot["mae_point_um"] - bm_plot["mae_ci_low_um"], bm_plot["mae_ci_high_um"] - bm_plot["mae_point_um"]],
    fmt="o", capsize=3, color="#1f77b4"
)
ax.set_yticks(y_pos)
ax.set_yticklabels(bm_plot["model_id"])
ax.invert_yaxis()
ax.set_xlabel("Raw MAE (um) with Exploratory 95% Image-Cluster Bootstrap CI")
ax.set_title(f"Exploratory Model MAE 95% CIs (B=10,000, n_clusters={bm_plot['n_clusters'].iloc[0]}, {RUN_ID})")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig05_bootstrap_mae_intervals.png")
plt.show()

bp_ml01 = boot_p[(boot_p["comparator_id"] == "ML01") & (boot_p["candidate_id"].isin(order_ids))].set_index("candidate_id").loc[[m for m in order_ids if m != "ML01"]].reset_index()
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(bp_ml01))
ax.errorbar(
    bp_ml01["delta_mae_point_um"], y_pos,
    xerr=[bp_ml01["delta_mae_point_um"] - bp_ml01["delta_mae_ci_low_um"], bp_ml01["delta_mae_ci_high_um"] - bp_ml01["delta_mae_point_um"]],
    fmt="s", capsize=3, color="#d62728"
)
ax.axvline(0.0, color="k", ls="--", lw=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(bp_ml01["candidate_id"])
ax.invert_yaxis()
ax.set_xlabel("Paired Delta MAE vs ML01 (um) — Negative Favors Candidate")
ax.set_title(f"Exploratory Paired MAE Difference vs ML01 Extra Trees ({RUN_ID})")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig06_paired_delta_mae_forest_vs_ML01.png")
plt.show()

## 16–19. Zero-Width, Per-Series/Stage, Image-Group Heatmap & Tail-Error Diagnostics

In [ ]:
zw_df = pd.read_csv(RUN_DIR / "M04B_ZERO_WIDTH_ANALYSIS.csv")
sub_df = pd.read_csv(RUN_DIR / "M04B_SUBGROUP_METRICS.csv")
tail_df = pd.read_csv(RUN_DIR / "M04B_TAIL_ERROR_ANALYSIS.csv")
print("=== ZERO-WIDTH REGIME (n_zero_width_val=", zw_df["n_zero_width_val"].iloc[0], ") ===")
print(zw_df.set_index("model_id").loc[raw_m["model_id"].head(10)].to_string())

ig_piv = sub_df[(sub_df["subgroup_type"] == "ImageGroup") & (sub_df["model_id"].isin(raw_m["model_id"].head(10)))].pivot(
    index="subgroup_value", columns="model_id", values="raw_mae_um"
)[raw_m["model_id"].head(10)]
fig, ax = plt.subplots(figsize=(11, 8))
im = ax.imshow(ig_piv.to_numpy(), aspect="auto", cmap="YlOrRd")
ax.set_xticks(np.arange(len(ig_piv.columns)))
ax.set_xticklabels(ig_piv.columns, rotation=45, ha="right")
ax.set_yticks(np.arange(len(ig_piv.index)))
ax.set_yticklabels(ig_piv.index, fontsize=7)
plt.colorbar(im, ax=ax, label="Raw MAE (um)")
ax.set_title(f"Image-Group MAE Heatmap — Top-10 Models ({RUN_ID})")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig07_image_group_heatmap.png")
plt.show()

print("\n=== TOP-10 HARDEST VALIDATION OBSERVATIONS (TAIL ERROR) ===")
print(tail_df.head(10).to_string(index=False))

## 20–23. Computational Feasibility, Accuracy-Efficiency Pareto & Finalist Proposals

In [ ]:
res_df = pd.read_csv(RUN_DIR / "M04B_RESOURCE_USAGE.csv")
pareto = raw_m.merge(res_df[res_df["status"] == "PASS"][["model_id", "total_elapsed_sec", "device"]], on="model_id")
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(pareto["total_elapsed_sec"], pareto["mae_um"], s=60, alpha=0.8)
for _, row in pareto.iterrows():
    ax.annotate(row["model_id"], (row["total_elapsed_sec"], row["mae_um"]), fontsize=8, xytext=(4, 2), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("Total Elapsed Time (s, log scale) — Single-Run Screening")
ax.set_ylabel("Raw Validation MAE (um)")
ax.set_title(f"M04B Accuracy vs Runtime Pareto ({RUN_ID})")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig08_accuracy_efficiency_pareto.png")
plt.show()